# Практическая работа №2. Часть 4А
## Аугментация данных: сдвиг, отражение, поворот, яркость и стирание

**Цель:** написать своими руками преобразования картинок, которые не меняют класс: сдвиг, отражение, поворот с билинейной интерполяцией, изменение яркости и контраста, случайное стирание; сверить их с `torchvision`, собрать из них конвейер аугментации на лету и проверить, как он меняет точность и устойчивость сверточной сети на маленькой обучающей выборке.

> **Правило части.** В заданиях А1–А3 преобразования пишутся на тензорах: срезы, арифметика и `F.grid_sample` для выборки значений по сетке координат. Функции `torchvision.transforms.functional` нужны только для сверки. В А4 конвейер аугментации пишется как собственный класс из ваших функций, готовые `transforms.RandomAffine`, `RandomErasing` и подобные не используются.

**Задача:** классификация одежды FashionMNIST на той же маленькой обучающей выборке из 1000 картинок, что и в части 3Р; валидационная и тестовая выборки по 2000 картинок. Сеть `WideCNN` – это `SimpleCNN` из части 4А первой практической работы, расширенная до 16 и 32 каналов и с дополнительным полносвязным слоем на 128 нейронов (около 207 тысяч параметров). Сквозные учебные примеры со слайдов: сдвиг картинки $3 \times 3$, поворот на 90° и билинейная интерполяция в точке $(0{,}25;\ 0{,}5)$, яркость и контраст пикселя $0{,}6$ при среднем $0{,}2$, размер прямоугольника стирания.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Там, где результат зависит от случайных чисел, указано примерное значение или качественное ожидание. В ноутбуке четыре обучения сети по 100 эпох, каждое занимает на процессоре 1–3 минуты. Сначала прочитайте теорию ниже.

## Как устроена аугментация данных

**1. Больше данных – лучшая регуляризация.** Гудфеллоу начинает раздел 7.4 так: самый лучший способ повысить обобщающую способность модели – обучить ее на большем объеме данных. Новые данные с разметкой дороги, но для классификации их можно получить почти бесплатно: если преобразование $T$ не меняет класс картинки, то пара $(T(x), y)$ – новый правильный пример. Такой прием называют **аугментацией** данных (у Гудфеллоу – пополнение набора данных). Добавляя в обучение сдвинутые, повернутые и затемненные картинки с теми же метками, мы сообщаем сети, что от этих изменений ответ зависеть не должен. В Б2.3 регуляризация уменьшала разрыв между выборками, но точность на 1000 картинках упиралась примерно в 0,81; аугментация – способ увеличить сами данные.

**2. Что сверточная сеть умеет сама, а что нет.** Свертка эквивариантна к сдвигу: если сдвинуть вход, карта признаков сдвинется так же (Б1.3). Пулинг $2 \times 2$ дает устойчивость к сдвигам примерно на 1 пиксель внутри окна. Но полносвязный слой в конце сети привязан к положению признаков на карте, поэтому сдвиг на 3–4 пикселя меняет вход этого слоя почти полностью. К повороту, масштабу, яркости и контрасту свертка не дает никакой устойчивости. В FashionMNIST все вещи аккуратно отцентрованы и одинаково ориентированы, поэтому сеть, обученная без аугментации, никогда не видела сдвинутых и повернутых картинок. В задании А1 это видно на опыте.

**3. Сдвиг и отражение.** Сдвиг на $(d_x, d_y)$ пикселей: $y[i, j] = x[i - d_y,\ j - d_x]$, а пиксели, которым не нашлось источника, заполняются нулем (черным фоном). Положительное $d_x$ сдвигает картинку вправо, положительное $d_y$ – вниз, как в `torchvision`. Пример: картинка $3 \times 3$ со строками $(1, 2, 3)$, $(4, 5, 6)$, $(7, 8, 9)$ после сдвига на $d_x = 1$ дает строки $(0, 1, 2)$, $(0, 4, 5)$, $(0, 7, 8)$. На тензорах сдвиг – это копирование среза в срез: `out[..., 0:3, 1:3] = img[..., 0:3, 0:2]`. Горизонтальное отражение – разворот последней оси: `img.flip(-1)`, строка $(1, 2, 3)$ становится $(3, 2, 1)$.

**4. Поворот и билинейная интерполяция.** При повороте пиксели попадают между узлами сетки, поэтому поворот считают **в обратную сторону**: для каждого пикселя результата находят точку исходной картинки, из которой он пришел, и берут значение в этой точке. Координаты отсчитывают от центра картинки: для пикселя в строке $i$ и столбце $j$
$$x = j - \frac{W-1}{2}, \qquad y = i - \frac{H-1}{2},$$
у картинки $28 \times 28$ центр находится в точке $(13{,}5;\ 13{,}5)$. Ось $y$ направлена вниз. Для поворота на угол $\theta$ против часовой стрелки (так поворачивает `torchvision` при $\theta > 0$) точка-источник равна
$$x_{\text{ист}} = x\cos\theta - y\sin\theta, \qquad y_{\text{ист}} = x\sin\theta + y\cos\theta.$$
*Пример.* $\theta = 90°$, $\cos\theta = 0$, $\sin\theta = 1$: пиксель результата слева от центра, $(x, y) = (-1, 0)$, берет значение из точки $(0, -1)$, то есть сверху от центра. Верх картинки ушел влево – это и есть поворот против часовой стрелки.

Точка-источник обычно лежит между четырьмя пикселями. **Билинейная интерполяция** берет их значения с весами, которые тем больше, чем ближе пиксель: если точка отстоит от левого верхнего пикселя на $f_x$ по горизонтали и $f_y$ по вертикали ($0 \le f_x, f_y < 1$), то
$$v = (1-f_x)(1-f_y)\,p_{00} + f_x(1-f_y)\,p_{01} + (1-f_x)f_y\,p_{10} + f_x f_y\,p_{11}.$$
Пример: $p_{00} = 0$, $p_{01} = 1$, $p_{10} = 0{,}5$, $p_{11} = 1$, $f_x = 0{,}25$, $f_y = 0{,}5$: $v = 0{,}375 \cdot 0 + 0{,}125 \cdot 1 + 0{,}375 \cdot 0{,}5 + 0{,}125 \cdot 1 = 0{,}4375$. Если точка вышла за край картинки, значение равно 0.

В PyTorch выборку по сетке делает `F.grid_sample(img, grid, mode='bilinear', padding_mode='zeros', align_corners=False)`. Сетка `grid` формы $(N, H, W, 2)$ задает для каждого пикселя результата точку-источник в **нормированных** координатах от $-1$ до $1$: при `align_corners=False` нормированная координата равна $u = x_{\text{ист}} / (W/2)$, $v = y_{\text{ист}} / (H/2)$, где $x_{\text{ист}}, y_{\text{ист}}$ отсчитаны от центра, как выше. Значение $-1$ соответствует левому краю левого пикселя, $+1$ – правому краю правого.

**5. Яркость, контраст и шум.** Для черно-белой картинки со значениями от 0 до 1:
- яркость: $y = \mathrm{clip}(f \cdot x,\ 0,\ 1)$, при $f < 1$ картинка темнее;
- контраст: $y = \mathrm{clip}(c \cdot x + (1-c)\cdot m,\ 0,\ 1)$, где $m$ – среднее значение пикселей этой картинки; при $c < 1$ значения стягиваются к среднему;
- шум: $y = \mathrm{clip}(x + \varepsilon,\ 0,\ 1)$ со случайным $\varepsilon$ (раздел 7.5: привнесение шума во входные данные – тоже аугментация).

Пример: пиксель $0{,}6$ при $f = 1{,}5$ дает $0{,}9$, при $f = 2$ упирается в 1. Контраст $c = 0{,}5$ при среднем $m = 0{,}2$: $0{,}5 \cdot 0{,}6 + 0{,}5 \cdot 0{,}2 = 0{,}4$. Это те же формулы, что в `TF.adjust_brightness` и `TF.adjust_contrast`. Для цветных картинок добавляют насыщенность и цветовой тон (`ColorJitter`), а Крижевский и соавторы (2012) случайно сдвигали цвета вдоль главных компонент.

**6. Случайное стирание.** Случайное стирание (Чжун и соавторы, 2017; близкий метод cutout, Деврис и Тейлор, 2017) закрашивает на картинке прямоугольник случайного размера и положения. Площадь прямоугольника составляет случайную долю $s$ площади картинки, отношение сторон $r$ тоже случайно; высота и ширина
$$h = \mathrm{round}\big(\sqrt{s \cdot H W \cdot r}\big), \qquad w = \mathrm{round}\big(\sqrt{s \cdot H W / r}\big).$$
Пример: $s = 0{,}1$, $r = 1$: $0{,}1 \cdot 784 = 78{,}4$, $h = w = \mathrm{round}(8{,}85) = 9$. Сеть учится узнавать вещь, даже если часть ее закрыта, и не может опираться на один маленький участок. Это похоже на прореживание из Б2.3, только выключаются соседние пиксели входа.

**7. Какие преобразования допустимы.** Преобразование допустимо, только если оно не меняет правильный ответ. Гудфеллоу (раздел 7.4) приводит пример: для распознавания символов нельзя отражать картинки относительно вертикальной оси и поворачивать на 180°, иначе «b» станет «d», а «6» станет «9». Другие примеры:
- текст и цифры: отражение недопустимо, небольшой поворот и сдвиг допустимы;
- дорожные знаки: отражение превращает «поворот налево» в «поворот направо»; такое преобразование можно использовать, только если поменять и метку;
- сигналы светофора: менять цветовой тон нельзя, ведь класс задан цветом;
- FashionMNIST: горизонтальное отражение допустимо, одежда почти симметрична; переворот вверх ногами не меняет название вещи, но таких картинок в настоящих данных нет, и сеть тратит на них свои возможности (задание А5).

Сила преобразований тоже важна: поворот на 10° делает картинку правдоподобнее, поворот на 90° уже нет. Слишком сильная аугментация приводит к недообучению.

**8. Где и когда применять.** Аугментация применяется **только к обучающей выборке** и **на лету**: каждый раз, когда картинка попадает в мини-пакет, к ней применяются новые случайные преобразования. За 100 эпох сеть увидит 100 разных вариантов каждой картинки, а заранее размноженный в 10 раз набор дал бы только 10. Валидационную и тестовую выборки не меняют: они должны изображать настоящие новые данные. Отдельная идея – **аугментация при проверке**: показать сети несколько вариантов одной тестовой картинки (например, саму картинку и ее отражение) и усреднить вероятности (Гудфеллоу, раздел 12.2.1). При сравнении двух моделей обе должны обучаться с одной и той же схемой аугментации, иначе неизвестно, что дало выигрыш (раздел 7.4).

**9. Смешивание картинок.** Метод mixup (Чжан и соавторы, 2018) обучает сеть на смесях пар картинок: $\tilde x = \lambda x_a + (1-\lambda)x_b$, $\tilde y = \lambda y_a + (1-\lambda)y_b$, где $y$ – вектор вероятностей классов, а $\lambda$ – случайное число из бета-распределения. При $\lambda = 0{,}7$ цель равна $0{,}7$ для класса первой картинки и $0{,}3$ для второй; как и сглаживание меток из Б2.3, это запрещает сети быть уверенной на 100 %. В методе CutMix (Юн и соавторы, 2019) вместо смешивания всей картинки вставляют прямоугольник из другой картинки, а метки смешивают пропорционально площади. Mixup – задание повышенной сложности.

**10. Звук и временные ряды.** В Теме 2 сети будут работать со звуком и последовательностями, и там аугментация тоже есть (раздел 7.4 упоминает распознавание речи): сдвиг сигнала во времени, добавление шума, изменение громкости, небольшое растяжение по времени. Для спектрограмм используют SpecAugment (Пак и соавторы, 2019): закрывают случайные полосы частот и отрезки времени, то есть то же случайное стирание, но на картинке «время × частота». Для показаний датчиков (скорость, ускорение) – шум, масштаб и сдвиг окна.

**11. В PyTorch.** В `torchvision.transforms` есть готовые классы: `RandomAffine(degrees, translate)`, `RandomHorizontalFlip`, `RandomRotation`, `ColorJitter`, `RandomErasing`; их собирают в `transforms.Compose([...])` и передают набору данных параметром `transform=` только для обучающей выборки. В этой работе мы пишем те же преобразования сами и применяем их к целому мини-пакету.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import math
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

# Цвета графиков – как на слайдах
NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# FashionMNIST заранее скачивается в папку data (интернет в аудитории не нужен):
#   datasets.FashionMNIST('data', train=True, download=True); datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
tf = transforms.ToTensor()
try:
    train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=False, transform=tf)
except Exception:
    print('FashionMNIST не найден в папке data – используются случайные картинки FakeData, ожидаемые числа не совпадут')
    train_ds = datasets.FakeData(size=60000, image_size=(1, 28, 28), num_classes=10, transform=tf)
    test_ds = datasets.FakeData(size=2000, image_size=(1, 28, 28), num_classes=10, transform=tf, random_offset=60000)

def take(ds, idx):
    """Картинки и метки с номерами idx одним тензором: выборка маленькая и целиком помещается в память."""
    idx = list(idx)
    if hasattr(ds, 'data'):
        return ds.data[idx].float().div(255).unsqueeze(1), ds.targets[idx].clone()
    xs, ys = zip(*(ds[i] for i in idx))
    return torch.stack(xs), torch.as_tensor([int(y) for y in ys])

X_train, y_train = take(train_ds, range(1000))            # обучающая выборка: те же 1000 картинок, что в части 3Р
X_val, y_val = take(train_ds, range(58000, 60000))        # валидационная: последние 2000 картинок обучающего набора
X_test, y_test = take(test_ds, range(2000))               # тестовая: первые 2000 картинок тестового набора
CLASSES = ['футболка', 'брюки', 'пуловер', 'платье', 'пальто', 'сандалия', 'рубашка', 'кроссовка', 'сумка', 'ботинок']
print(tuple(X_train.shape), tuple(X_val.shape), tuple(X_test.shape), '  ожидается (1000, 1, 28, 28) (2000, 1, 28, 28) (2000, 1, 28, 28)')

# SimpleCNN из части 4А ПР1, расширенная: 16 и 32 канала и скрытый полносвязный слой на 128 нейронов
class WideCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

count = lambda m: sum(p.numel() for p in m.parameters())
print(count(WideCNN()), '  ожидается 206922')

def evaluate(model, X, y, batch_size=500):
    """Средняя потеря и точность в режиме применения."""
    model.eval()
    loss, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            logits = model(X[i:i + batch_size])
            loss += F.cross_entropy(logits, y[i:i + batch_size], reduction='sum').item()
            correct += (logits.argmax(dim=1) == y[i:i + batch_size]).sum().item()
    return loss / len(X), correct / len(X)

def fit(model, epochs=100, batch_size=50, augment=None, lr=1e-3):
    """Обучение Adam по мини-пакетам; augment(x) – аугментация на лету, только для обучающих пакетов."""
    g = torch.Generator().manual_seed(RANDOM_STATE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(X_train), generator=g)
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            x = X_train[idx]
            if augment is not None:
                x = augment(x)
            opt.zero_grad()
            F.cross_entropy(model(x), y_train[idx]).backward()
            opt.step()
        tl, ta = evaluate(model, X_train, y_train)      # обучающая выборка без аугментации
        vl, va = evaluate(model, X_val, y_val)
        for key, value in zip(hist, [tl, ta, vl, va]):
            hist[key].append(value)
    return hist

results, models = {}, {}

def train_variant(name, augment=None, epochs=100):
    """Одно обучение WideCNN с одними и теми же начальными весами; итоги складываются в results и models."""
    start = time.perf_counter()
    torch.manual_seed(RANDOM_STATE)
    model = WideCNN()
    hist = fit(model, epochs=epochs, augment=augment)
    _, te = evaluate(model, X_test, y_test)
    results[name] = dict(train_acc=hist['train_acc'][-1], val_acc=hist['val_acc'][-1], val_loss=hist['val_loss'][-1],
                         best_val_loss=min(hist['val_loss']), test_acc=te, hist=hist)
    models[name] = model
    print(f'{name:<26} обуч. {hist["train_acc"][-1]:.3f}   валид. {hist["val_acc"][-1]:.3f} '
          f'(потеря {hist["val_loss"][-1]:.3f}, лучшая {min(hist["val_loss"]):.3f})   тест {te:.3f}   [{time.perf_counter() - start:.0f} с]')
    return model

def show_images(rows, titles, fname, row_labels=None):
    """Сетка картинок: rows – список тензоров формы (N, 1, 28, 28) одинаковой длины."""
    n_rows, n_cols = len(rows), len(rows[0])
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(1.3 * n_cols, 1.4 * n_rows), squeeze=False)
    for r, imgs in enumerate(rows):
        for c in range(n_cols):
            ax = axs[r, c]
            ax.imshow(imgs[c, 0], cmap='gray_r', vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0 and titles:
                ax.set_title(titles[c], fontsize=8)
            if c == 0 and row_labels:
                ax.set_ylabel(row_labels[r], fontsize=8)
    fig.tight_layout(); fig.savefig(fname, dpi=100); plt.close(fig)
    display(Image(fname))

**Задание А1. Сдвиг и отражение.**

- `shift(img, dx, dy)` сдвигает картинку (тензор формы `(..., H, W)`, то есть одну картинку или пакет) на `dx` пикселей вправо и `dy` вниз, освободившиеся пиксели заполняются нулем (п. 3 теории). Сдвиги могут быть отрицательными. Удобно сначала создать `out = torch.zeros_like(img)`, а потом скопировать срез в срез; границы срезов уже посчитаны в заготовке для оси `x`, по образцу допишите ось `y`.
- `hflip(img)` – отражение слева направо: разворот последней оси.

*Проверка по п. 3 теории.* Картинка $3 \times 3$ с числами от 1 до 9 после сдвига на $d_x = 1$ дает строки $(0, 1, 2)$, $(0, 4, 5)$, $(0, 7, 8)$, после сдвига на $d_y = -1$ – строки $(4, 5, 6)$, $(7, 8, 9)$, $(0, 0, 0)$. На настоящих картинках ваш сдвиг совпадает с `TF.affine(..., translate=[dx, dy])`, а отражение с `TF.hflip`. Затем готовая ячейка обучает `WideCNN` без аугментации и проверяет ее на тестовых картинках, сдвинутых на 1–4 пикселя.

In [ ]:
def shift(img, dx, dy):
    H, W = img.shape[-2:]
    out = torch.zeros_like(img)
    xs, xd = (slice(0, W - dx), slice(dx, W)) if dx >= 0 else (slice(-dx, W), slice(0, W + dx))   # откуда и куда по x
    ys, yd = ...                                    # <-- то же по y
    out[..., yd, xd] = ...                          # <-- скопировать срез исходной картинки на новое место
    return out

def hflip(img):
    return ...                                      # <-- развернуть последнюю ось


tiny = torch.arange(1.0, 10.0).view(1, 3, 3)
print(show(shift(tiny, 1, 0)[0]), '  ожидается [[0.0, 1.0, 2.0], [0.0, 4.0, 5.0], [0.0, 7.0, 8.0]]')
print(show(shift(tiny, 0, -1)[0]), '  ожидается [[4.0, 5.0, 6.0], [7.0, 8.0, 9.0], [0.0, 0.0, 0.0]]')
print(show(hflip(tiny)[0, 0]), '  ожидается [3.0, 2.0, 1.0]')

batch = X_test[:16]
diff_shift = max((shift(batch, dx, dy) - TF.affine(batch, angle=0.0, translate=[dx, dy], scale=1.0, shear=[0.0, 0.0])).abs().max().item()
                 for dx, dy in [(3, 0), (-2, 1), (0, -4)])
print(diff_shift < 1e-6, (hflip(batch) - TF.hflip(batch)).abs().max().item() == 0,
      '  ожидается True True (совпадение с torchvision)')
show_images([batch[:6], shift(batch[:6], 3, 0), shift(batch[:6], 0, -4), hflip(batch[:6])], None, 'shift_flip.png',
            row_labels=['исходные', 'dx = 3', 'dy = -4', 'отражение'])

In [ ]:
# Готовая ячейка: WideCNN без аугментации (100 эпох) и проверка на сдвинутых тестовых картинках
train_variant('без аугментации')

def shift_curve(model, max_shift=4):
    """Точность на тестовой выборке, сдвинутой на s пикселей вправо и вниз (среднее по двум направлениям)."""
    accs = [evaluate(model, X_test, y_test)[1]]
    for s in range(1, max_shift + 1):
        a_x = evaluate(model, shift(X_test, s, 0), y_test)[1]
        a_y = evaluate(model, shift(X_test, 0, s), y_test)[1]
        accs.append((a_x + a_y) / 2)
    return accs

acc_shift_base = shift_curve(models['без аугментации'])
print('точность при сдвиге на 0, 1, 2, 3, 4 пикселя:', [round(a, 3) for a in acc_shift_base])
print('  ожидается: без сдвига около 0.84; при сдвиге на 2 пикселя около 0.68, на 3 около 0.53, на 4 около 0.4')

**Вопрос А1.** а) Почему при сдвиге на 1 пиксель точность падает мало, а при сдвиге на 3–4 пикселя сильно, хотя свертка эквивариантна к сдвигу? б) Почему в FashionMNIST сдвиги почти не встречаются, а в задаче «найти вещь на фотографии с камеры» встречаются постоянно? в) Почему после сдвига освободившиеся пиксели заполняют нулем, а не переносят с другого края, как делает `torch.roll`?

*Ваш ответ:*

**Задание А2. Поворот с билинейной интерполяцией.**

- `rotation_grid(angles, H, W)` строит сетку для `F.grid_sample` по п. 4 теории. `angles` – тензор углов в градусах формы `(N,)`, у каждой картинки пакета свой угол. Заготовка уже содержит координаты центров пикселей результата `x`, `y` (отсчитаны от центра картинки, форма `(1, H, W)`) и косинусы и синусы углов формы `(N, 1, 1)`. Нужно найти точку-источник $x_{\text{ист}} = x\cos\theta - y\sin\theta$, $y_{\text{ист}} = x\sin\theta + y\cos\theta$ и перевести ее в нормированные координаты делением на $W/2$ и $H/2$. Результат – `torch.stack([u, v], dim=-1)` формы `(N, H, W, 2)`.
- `rotate(imgs, angles)` поворачивает пакет `(N, 1, H, W)`: `F.grid_sample(imgs, grid, mode='bilinear', padding_mode='zeros', align_corners=False)`.

*Проверка по п. 4 теории.* На картинке $3 \times 3$ с единицей в середине верхней строки поворот на 90° переносит единицу в середину левого столбца. При угле 17° ваш поворот совпадает с `TF.rotate(..., interpolation=InterpolationMode.BILINEAR)` с точностью до округлений. Затем готовая ячейка проверяет сеть из А1 на повернутых тестовых картинках.

In [ ]:
def rotation_grid(angles, H, W):
    theta = torch.deg2rad(angles.float()).view(-1, 1, 1)                 # (N, 1, 1)
    cos, sin = torch.cos(theta), torch.sin(theta)
    x = (torch.arange(W).float() - (W - 1) / 2).view(1, 1, W).expand(1, H, W)   # координаты от центра, ось x вправо
    y = (torch.arange(H).float() - (H - 1) / 2).view(1, H, 1).expand(1, H, W)   # ось y вниз
    x_src = ...                                     # <-- откуда брать значение: x
    y_src = ...                                     # <-- откуда брать значение: y
    u = ...                                         # <-- нормированная координата x
    v = ...                                         # <-- нормированная координата y
    return torch.stack([u, v], dim=-1)

def rotate(imgs, angles):
    grid = rotation_grid(angles, imgs.shape[-2], imgs.shape[-1])
    return ...                                      # <-- выборка по сетке


dot = torch.zeros(1, 1, 3, 3)
dot[0, 0, 0, 1] = 1.0                                     # единица в середине верхней строки
print(show(rotate(dot, torch.tensor([90.0]))[0, 0]), '  ожидается [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]]')

ref = torch.stack([TF.rotate(img, 17.0, interpolation=InterpolationMode.BILINEAR) for img in batch])
mine = rotate(batch, torch.full((len(batch),), 17.0))
print((mine - ref).abs().max().item() < 1e-4, '  ожидается True (совпадение с TF.rotate)')

angles6 = torch.tensor([-30.0, -10.0, 0.0, 10.0, 30.0, 90.0])
show_images([rotate(batch[:1].expand(6, -1, -1, -1), angles6)], [f'{a:.0f}°' for a in angles6], 'rotations.png')

In [ ]:
# Готовая ячейка: сеть без аугментации на повернутых тестовых картинках
def rotation_curve(model, angles=(0, 10, 20, 30)):
    """Точность на тестовой выборке, повернутой на угол a (среднее по +a и -a)."""
    accs = []
    for a in angles:
        plus = evaluate(model, rotate(X_test, torch.full((len(X_test),), float(a))), y_test)[1]
        minus = evaluate(model, rotate(X_test, torch.full((len(X_test),), -float(a))), y_test)[1]
        accs.append((plus + minus) / 2)
    return accs

acc_rot_base = rotation_curve(models['без аугментации'])
print('точность при повороте на 0, 10, 20, 30 градусов:', [round(a, 3) for a in acc_rot_base])
print('  ожидается: около 0.84, 0.75, 0.42, 0.22: поворот на 20° отнимает у сети половину точности')

**Вопрос А2.** а) Почему поворот считают «в обратную сторону», от пикселя результата к точке исходной картинки? Что случилось бы, если бы мы переносили каждый исходный пиксель на новое место? б) Почему углы в задании маленькие, а поворот на 90° для FashionMNIST вреден? в) Почему поворот на 90° картинки $3 \times 3$ получился точным, без размытия, а поворот на 17° размывает картинку?

*Ваш ответ:*

**Задание А3. Яркость, контраст и случайное стирание.**

- `adjust_brightness(img, f)` – $\mathrm{clip}(f \cdot x,\ 0,\ 1)$ (п. 5 теории), обрезка через `.clamp(0, 1)`.
- `adjust_contrast(img, c)` – $\mathrm{clip}(c \cdot x + (1 - c)\cdot m,\ 0,\ 1)$, где $m$ – среднее по пикселям **каждой** картинки: `img.mean(dim=(-3, -2, -1), keepdim=True)`.
- `erase(img, i, j, h, w)` – копия картинки, в которой прямоугольник с левым верхним углом в строке `i`, столбце `j`, высотой `h` и шириной `w` заполнен нулем.
- `erase_size(s, r, H, W)` – высота и ширина прямоугольника по п. 6: `round(sqrt(s * H * W * r))` и `round(sqrt(s * H * W / r))`.

*Проверка по п. 5–6 теории.* Пиксель $0{,}6$: яркость $f = 1{,}5$ дает $0{,}9$, $f = 2$ дает $1$; контраст $c = 0{,}5$ на картинке со средним $0{,}2$ дает $0{,}4$. Размер стирания при $s = 0{,}1$, $r = 1$: $(9, 9)$; при $s = 0{,}1$, $r = 2$: $(13, 6)$. На настоящих картинках функции совпадают с `TF.adjust_brightness`, `TF.adjust_contrast` и `TF.erase`.

In [ ]:
def adjust_brightness(img, f):
    return ...                                      # <-- умножить на f и обрезать в [0, 1]

def adjust_contrast(img, c):
    m = ...                                         # <-- среднее каждой картинки
    return ...                                      # <-- стянуть к среднему и обрезать

def erase(img, i, j, h, w):
    out = img.clone()
    ...                                             # <-- закрасить прямоугольник нулем
    return out

def erase_size(s, r, H=28, W=28):
    h = ...                                         # <-- высота
    w = ...                                         # <-- ширина
    return h, w


px = torch.tensor([[[0.6, 0.0], [0.2, 0.0]]])               # одна картинка 2 x 2, среднее 0.2
print(show(adjust_brightness(px, 1.5)[0, 0, 0]), show(adjust_brightness(px, 2.0)[0, 0, 0]),
      show(adjust_contrast(px, 0.5)[0, 0, 0]), '  ожидается 0.9 1.0 0.4')
print(erase_size(0.1, 1.0), erase_size(0.1, 2.0), '  ожидается (9, 9) (13, 6)')

checks = [(adjust_brightness(batch, 0.6) - TF.adjust_brightness(batch, 0.6)).abs().max().item(),
          (adjust_brightness(batch, 1.4) - TF.adjust_brightness(batch, 1.4)).abs().max().item(),
          (adjust_contrast(batch, 0.5) - TF.adjust_contrast(batch, 0.5)).abs().max().item(),
          (adjust_contrast(batch, 1.8) - TF.adjust_contrast(batch, 1.8)).abs().max().item(),
          (erase(batch, 5, 8, 9, 6) - TF.erase(batch, 5, 8, 9, 6, 0.0)).abs().max().item()]
print(max(checks) < 1e-6, '  ожидается True (совпадение с torchvision)')
show_images([batch[:6], adjust_brightness(batch[:6], 0.5), adjust_contrast(batch[:6], 0.4), erase(batch[:6], 8, 8, 9, 9)],
            None, 'color_erase.png', row_labels=['исходные', 'яркость 0.5', 'контраст 0.4', 'стирание'])

**Вопрос А3.** а) Чем отличается картинка с яркостью $0{,}5$ от картинки с контрастом $0{,}5$? б) Почему у FashionMNIST изменение яркости почти не помогает, а для фотографий с камеры на улице помогает сильно? в) Почему случайное стирание похоже на прореживание из Б2.3 и чем от него отличается?

*Ваш ответ:*

**Задание А4. Конвейер аугментации (главное задание).**

Допишите класс `Augment`. Вызов `augment(x)` получает обучающий пакет формы `(N, 1, 28, 28)` и возвращает новый пакет той же формы, где к **каждой** картинке применены свои случайные преобразования (п. 8 теории):
1. сдвиг на случайные целые `dx`, `dy` от `-max_shift` до `max_shift` (для каждой картинки свои, функция `shift` из А1);
2. отражение `hflip` с вероятностью `p_flip` (маска картинок `torch.rand(N) < p_flip`);
3. поворот `rotate` из А2 на случайный угол, равномерно распределенный от `-max_angle` до `max_angle`;
4. стирание с вероятностью `p_erase`: доля площади `s` от 0,02 до 0,2, отношение сторон `r` равномерно по логарифму от 0,3 до 3,3, размер по `erase_size`, положение случайное (готовая функция `random_box`).

Случайные числа берутся из генератора `self.g`, чтобы опыт повторялся. Затем готовые ячейки показывают 8 вариантов одной картинки, обучают сеть с аугментацией и сравнивают ее с сетью из А1 на обычных, сдвинутых и повернутых тестовых картинках.

In [ ]:
def random_box(g, H=28, W=28, s_range=(0.02, 0.2), r_range=(0.3, 3.3)):
    """Готовая функция: случайный прямоугольник стирания (i, j, h, w) по п. 6 теории."""
    s = s_range[0] + (s_range[1] - s_range[0]) * torch.rand(1, generator=g).item()
    log_r = math.log(r_range[0]) + (math.log(r_range[1]) - math.log(r_range[0])) * torch.rand(1, generator=g).item()
    h, w = erase_size(s, math.exp(log_r), H, W)
    h, w = min(h, H), min(w, W)
    i = torch.randint(0, H - h + 1, (1,), generator=g).item()
    j = torch.randint(0, W - w + 1, (1,), generator=g).item()
    return i, j, h, w

class Augment:
    def __init__(self, max_shift=2, p_flip=0.5, max_angle=10.0, p_erase=0.5, seed=RANDOM_STATE):
        self.max_shift, self.p_flip, self.max_angle, self.p_erase = max_shift, p_flip, max_angle, p_erase
        self.g = torch.Generator().manual_seed(seed)

    def __call__(self, x):
        x = x.clone()
        N = len(x)
        if self.max_shift > 0:
            d = torch.randint(-self.max_shift, self.max_shift + 1, (N, 2), generator=self.g)
            for k in range(N):
                x[k] = ...                          # <-- свой сдвиг для каждой картинки
        if self.p_flip > 0:
            flip = torch.rand(N, generator=self.g) < self.p_flip
            x[flip] = ...                           # <-- отразить выбранные картинки
        if self.max_angle > 0:
            angles = ...                            # <-- углы от -max_angle до max_angle
            x = ...                                 # <-- повернуть весь пакет
        if self.p_erase > 0:
            for k in range(N):
                if torch.rand(1, generator=self.g).item() < self.p_erase:
                    x[k] = ...                      # <-- стереть случайный прямоугольник
        return x


aug = Augment()
out = aug(X_train[:50])
print(tuple(out.shape), out.min().item() >= 0, out.max().item() <= 1, '  ожидается (50, 1, 28, 28) True True')
print((out - X_train[:50]).abs().amax(dim=(1, 2, 3)).gt(0).float().mean().item() > 0.9, '  ожидается True (изменены почти все картинки)')
print(torch.equal(aug(X_train[:5]), aug(X_train[:5])), '  ожидается False (при каждом вызове новые преобразования)')
show_images([Augment()(X_train[:1].expand(8, -1, -1, -1).clone())], [f'вариант {k + 1}' for k in range(8)], 'augment_variants.png')

In [ ]:
# Готовая ячейка: та же сеть с легкой и с сильной аугментацией на лету (по 100 эпох)
train_variant('легкая аугментация', augment=Augment(max_shift=1, p_flip=0.5, max_angle=0, p_erase=0))
train_variant('сильная аугментация', augment=Augment(max_shift=2, p_flip=0.5, max_angle=10, p_erase=0.5))
curves = {'без аугментации': (acc_shift_base, acc_rot_base)}
for name in ['легкая аугментация', 'сильная аугментация']:
    curves[name] = (shift_curve(models[name]), rotation_curve(models[name]))
for name, (cs, cr) in curves.items():
    print(f'{name:<22} сдвиг 0–4: {[round(a, 3) for a in cs]}   поворот 0–30°: {[round(a, 3) for a in cr]}')
print('  ожидается: легкая аугментация – тест около 0.85, на 1–2 пункта выше, чем без аугментации; сильная – тест около 0.83,')
print('  зато при сдвиге на 3 пикселя около 0.77 вместо 0.53 и при повороте на 20° около 0.67 вместо 0.42;')
print('  валидационная потеря к концу обучения: без аугментации растет примерно до 1.07, с легкой около 0.63, с сильной около 0.49')

fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
colors = {'без аугментации': GRAY, 'легкая аугментация': DEEP, 'сильная аугментация': BLUE}
for name, c in colors.items():
    h = results[name]['hist']
    axs[0].plot(range(1, len(h['val_loss']) + 1), h['val_loss'], color=c, lw=2.2, label=name)
axs[0].set_xlabel('эпоха'); axs[0].set_ylabel('потеря на валидации'); axs[0].legend(fontsize=9)
for ax, k, xs, xlabel in [(axs[1], 0, range(5), 'сдвиг, пикселей'), (axs[2], 1, [0, 10, 20, 30], 'поворот, градусов')]:
    for name, c in colors.items():
        ax.plot(list(xs), curves[name][k], 'o-', color=c, lw=2.2, label=name)
    ax.set_xlabel(xlabel); ax.set_ylabel('точность на тесте'); ax.set_ylim(0, 1)
fig.tight_layout(); fig.savefig('augment_compare.png', dpi=100); plt.close(fig)
display(Image('augment_compare.png'))

**Вопрос А4.** а) Почему точность на обучающей выборке у сетей с аугментацией ниже, а валидационная потеря в конце обучения намного лучше? б) Почему аугментацию нельзя применять к валидационной выборке, по которой мы выбираем модель? в) Сильная аугментация не подняла точность на обычных тестовых картинках, а легкая подняла. Какую из двух сетей вы поставите на беспилотник с камерой и почему?

*Ваш ответ:*

**Задание А5. Недопустимая аугментация и аугментация при проверке.**

- `vflip(img)` – отражение сверху вниз: разворот предпоследней оси.
- `tta_predict(model, x)` – аугментация при проверке (п. 8 теории): вероятности классов `F.softmax(model(x), dim=1)` для самих картинок и для их отражений `hflip(x)`, усредненные; модель в режиме `eval`, вычисления внутри `torch.no_grad()`.

*Проверка.* `vflip` переворачивает строки картинки $3 \times 3$: первая строка становится $(7, 8, 9)$. Сумма вероятностей `tta_predict` по классам равна 1. Затем готовая ячейка обучает сеть с легкой аугментацией и недопустимым для этой задачи переворотом вверх ногами и сравнивает все сети на тесте с аугментацией при проверке и без нее.

In [ ]:
def vflip(img):
    return ...                                      # <-- развернуть предпоследнюю ось

def tta_predict(model, x):
    model.eval()
    with torch.no_grad():
        p = ...                                     # <-- вероятности для самих картинок
        p_flip = ...                                # <-- вероятности для отражений
    return (p + p_flip) / 2


print(show(vflip(tiny)[0, 0]), '  ожидается [7.0, 8.0, 9.0]')
probs = tta_predict(models['легкая аугментация'], X_test[:100])
print(torch.allclose(probs.sum(dim=1), torch.ones(100)), '  ожидается True')

In [ ]:
# Готовая ячейка: недопустимая аугментация и аугментация при проверке
class AugmentVFlip(Augment):
    """Легкая аугментация плюс переворот вверх ногами с вероятностью 0.5."""
    def __call__(self, x):
        x = super().__call__(x)
        flip = torch.rand(len(x), generator=self.g) < 0.5
        x[flip] = vflip(x[flip])
        return x

train_variant('легкая + вверх ногами', augment=AugmentVFlip(max_shift=1, p_flip=0.5, max_angle=0, p_erase=0))
print()
for name in ['без аугментации', 'легкая аугментация', 'сильная аугментация', 'легкая + вверх ногами']:
    model = models[name]
    plain = evaluate(model, X_test, y_test)[1]
    tta = (tta_predict(model, X_test).argmax(dim=1) == y_test).float().mean().item()
    print(f'{name:<28} тест {plain:.3f}   с отражением при проверке {tta:.3f}')
print('  ожидается: без аугментации около 0.84, но с отражением при проверке хуже, около 0.78; у сетей, обученных с отражениями,')
print('  отражение при проверке добавляет несколько десятых пункта; переворот вверх ногами снижает тест примерно с 0.85 до 0.81')

# какие классы страдают от переворота
with torch.no_grad():
    pred_a = models['легкая аугментация'](X_test).argmax(dim=1)
    pred_v = models['легкая + вверх ногами'](X_test).argmax(dim=1)
print('\nточность по классам: легкая аугментация -> легкая + вверх ногами')
for k, name in enumerate(CLASSES):
    m = y_test == k
    print(f'{name:<10} {(pred_a[m] == k).float().mean().item():.2f} -> {(pred_v[m] == k).float().mean().item():.2f}')
print('  ожидается: сильнее всего падают пальто (около 0.86 -> 0.6) и рубашка (около 0.57 -> 0.45)')

**Вопрос А5.** а) Почему переворот вверх ногами не меняет название вещи, но все равно ухудшает сеть? Для каких классов это заметнее? б) Почему аугментация при проверке помогает сети, обученной с отражениями, и почти бесполезна для сети, которая отражений не видела? в) Придумайте по одному допустимому и недопустимому преобразованию для распознавания дорожных знаков и для распознавания рукописных цифр.

*Ваш ответ:*

### Задание повышенной сложности (к части 4А)

1. **Mixup своими руками.** Напишите функцию `mixup(x, y, alpha)`, которая перемешивает пакет случайной перестановкой и возвращает $\tilde x = \lambda x + (1-\lambda)x_{\text{перест}}$ и смешанные цели $\tilde y = \lambda\,\mathrm{onehot}(y) + (1-\lambda)\,\mathrm{onehot}(y_{\text{перест}})$, где $\lambda$ – одно число из бета-распределения `torch.distributions.Beta(alpha, alpha)`. Потеря – перекрестная энтропия с вероятностными целями `-(y_mix * F.log_softmax(logits, dim=1)).sum(dim=1).mean()`. Обучите `WideCNN` с mixup ($\alpha = 0{,}2$ и $\alpha = 1$) и сравните точность и валидационную потерю с результатами А4. Почему при $\alpha = 1$ сеть учится медленнее?
2. **CutMix.** Вместо смешивания всей картинки вставьте в нее прямоугольник из другой картинки пакета (по `random_box`) и смешайте метки пропорционально площади. Сравните с mixup и случайным стиранием.
3. **Сила аугментации.** Обучите сеть с `max_shift` от 0 до 4 и `max_angle` от 0 до 30° (по одному параметру за раз) и постройте точность на обычной и на сдвинутой тестовой выборке. Где начинается недообучение?
4. **Сколько стоит аугментация в данных.** Обучите сеть без аугментации на 2000 и 4000 картинках и найдите, скольким дополнительным размеченным картинкам равносильна аугментация на 1000 картинках по точности на сдвинутом тесте.
5. **Аугментация временного ряда.** Сгенерируйте синусоиды с шумом двух классов (частота 1 и 1,5 на отрезке из 100 точек) и обучите маленькую одномерную сверточную сеть (`nn.Conv1d`) на 50 примерах. Добавьте аугментации: сдвиг по времени, масштаб амплитуды, шум, стирание отрезка. Как они меняют точность? Это мост к Теме 2.

In [ ]:
# Место для задания повышенной сложности